In [ ]:
# Imports and file list
import os
import numpy as np
import pandas as pd
import tifffile as tiff
from PIL import Image

from Volume_bleeding import process_cone_positions, find_first_black_pixel_slice
from Number_bleeding import vessel_seg, process_cone_positions_num
from model_3D_visualization import visualize_cone_pyvista, cropping_img

# Choose the files(path)
# Reslice 2 and 3 are excluded: their raw data is too noisy.
data_dir = 'mouse_data'          # sits next to this notebook
assert os.path.isdir(data_dir), f"{os.path.abspath(data_dir)} not found - run this notebook from the CF_v2 folder"

tiff_files = [os.path.join(data_dir, f'Reslice of {i}.tif') for i in [0, 1, 4, 5, 6, 7, 8, 9]]

# Create folders to save the output
os.makedirs("output_csv", exist_ok=True)
os.makedirs("output_images", exist_ok=True)

In [ ]:
# All Parameters Setting
x_center = 2200             # x position used for the 3-D figure only
y_center = 50               # insertion position along the thin axis (slab centre)
crop_margin_x = 80          # half-width of the rendered slab along x
crop_margin_y = 50          # half-width along y
pos_num = 100               # insertion sites per animal
output_img_type = "tif"     # png, tif, jpg...
depth_limit = 1000          # insertion depth in voxels (= um)

# Probe geometry, all in micrometres, matching the table in README.md.
# Dictionary keys are the short names used in output filenames; `label` is the
# full name for figures and tables.
#
# Each probe is a shank (nearer the cortical surface) plus a tip (deepest),
# each tapering linearly from its base to its top diameter. Shank length plus
# tip length is 1000 um for every probe.
#
# These are DIAMETERS, which is what every simulation function takes. Nothing
# needs converting anywhere.
ELECTRODES = {
    "CF": dict(label="Carbon Fiber 10 um",
               shank_length=840,     shank_base_d=8.4,  shank_top_d=8.4,
               tip_length=160,       tip_base_d=6.8,    tip_top_d=0.0),
    "FMA": dict(label="Microprobes FMA",
                shank_length=971.91, shank_base_d=25.0, shank_top_d=25.0,
                tip_length=28.09,    tip_base_d=25.0,   tip_top_d=0.0),
    "Shuttle": dict(label="Shuttle 25 um",
                    shank_length=990, shank_base_d=25.0, shank_top_d=25.0,
                    tip_length=10,    tip_base_d=25.0,   tip_top_d=25.0),
    "UEA": dict(label="Blackrock UEA",
                shank_length=950,    shank_base_d=90.0, shank_top_d=28.0,
                tip_length=50,       tip_base_d=28.0,   tip_top_d=3.0),
}

In [ ]:
# Segment the vessel for all files
#
# Build the connected-component labelling that the count metric needs, once
# per animal, and cache it. Cells 4-8 read the cache through
# `load_segmented_data` below; they never re-segment.
#
# Caching matters because this is the slow, memory-hungry step: labelling one
# 3000 x 5000 x 100 volume produces an int32 label image of about 6 GB, and
# the compressed .npz is roughly 15-20 MB per animal. The first run pays for
# all eight; later runs only load.

seg_save_dir = "./seg_cache"
os.makedirs(seg_save_dir, exist_ok=True)

for file_path in tiff_files:
    file_label = os.path.splitext(os.path.basename(file_path))[0]
    seg_path = os.path.join(seg_save_dir, f"{file_label}_seg.npz")

    if not os.path.exists(seg_path):
        print(f"Segmenting {file_label} ...")
        # Same transpose as everywhere else: stored (z, y, x) -> (z, x, y).
        img_data = tiff.imread(file_path)
        img_data = np.transpose(img_data, axes=(0, 2, 1)).astype(np.uint16)
        # min_size=2 drops only isolated single voxels (about 2,600 per
        # volume). connectivity=2 means a 3x3x3 structuring element, so
        # diagonal neighbours count as connected. `distance` is accepted but
        # unused: vessel_seg has its merge_labels step commented out.
        seg_result = vessel_seg(img_data, min_size=2, connectivity=2, distance=2)
        np.savez_compressed(seg_path, seg=seg_result)

    else:
        print(f"Loading cached segmentation: {file_label}")
        # NOTE: seg_result is overwritten on every iteration and never read
        # here. On a warm cache this loop only verifies that all eight files
        # exist - at the cost of loading and discarding ~6 GB each time.
        seg_result = np.load(seg_path)['seg']


# Load one animal's cached label image. This is what the per-electrode cells
# call; it raises rather than silently re-segmenting if the cache is missing.
def load_segmented_data(file_path, seg_dir="./seg_cache"):
    file_label = os.path.splitext(os.path.basename(file_path))[0]
    seg_path = os.path.join(seg_dir, f"{file_label}_seg.npz")
    if not os.path.exists(seg_path):
        raise FileNotFoundError(f"Segmentation for {file_label} not found.")
    return np.load(seg_path)['seg']

In [ ]:
# Run every probe over every animal
#
# This replaces what used to be one near-identical cell per probe. The body is
# unchanged, only the outer loop is new, so the numbers are the same.
#
# Per probe: both metrics at `pos_num` sites on each animal, two CSVs, and one
# 3-D figure per animal.

for short_name, cfg in ELECTRODES.items():
    print(f"=== {cfg['label']} ({short_name}) ===")
    area_frames, num_frames = [], []

    for file_path in tiff_files:
        print(f"Processing {file_path} ...")
        file_label = os.path.splitext(os.path.basename(file_path))[0]

        # Raw binary volume feeds the volume metric, the labelled volume the count.
        img_data = tiff.imread(file_path)
        img_data = np.transpose(img_data, axes=(0, 2, 1)).astype(np.uint16)
        img_data_seg = load_segmented_data(file_path)

        # Same geometry for both metrics.
        geom = dict(y_center=y_center,
                    shank_length=cfg["shank_length"],
                    shank_base_diameter=cfg["shank_base_d"],
                    shank_top_diameter=cfg["shank_top_d"],
                    tip_length=cfg["tip_length"],
                    tip_base_diameter=cfg["tip_base_d"],
                    tip_top_diameter=cfg["tip_top_d"],
                    depth_limit=depth_limit, pos_num=pos_num)

        areas = process_cone_positions(img_data, **geom)
        nums = process_cone_positions_num(img_data_seg, **geom)

        area_frames.append(pd.DataFrame({"file": file_label, "position": range(pos_num), "overlap_area": areas}))
        num_frames.append(pd.DataFrame({"file": file_label, "position": range(pos_num), "overlap_number": nums}))

        # 3-D figure. Illustrative only: one site (x_center), not all pos_num sites,
        # and it does not feed into the CSVs.
        start_slice = find_first_black_pixel_slice(img_data, x_center, y_center)
        cropped, cx, cy = cropping_img(img_data, x_center, y_center, crop_margin_x, crop_margin_y, start_slice)
        plotter = visualize_cone_pyvista(
            cropped, cx, cy,
            shank_length=cfg["shank_length"],
            shank_base_diameter=cfg["shank_base_d"], shank_top_diameter=cfg["shank_top_d"],
            tip_length=cfg["tip_length"],
            tip_base_diameter=cfg["tip_base_d"], tip_top_diameter=cfg["tip_top_d"],
            start_slice=start_slice, depth_limit=depth_limit, ui=0)

        img_path = os.path.join("output_images", f"{short_name}_{file_label}.{output_img_type}")
        plotter.screenshot(img_path)
        Image.open(img_path).save(img_path, dpi=(300, 300))   # stamp 300 dpi metadata

    # One CSV per metric, covering all animals for this probe.
    pd.concat(area_frames, ignore_index=True).to_csv(os.path.join("output_csv", f"Volume_{short_name}.csv"), index=False)
    pd.concat(num_frames, ignore_index=True).to_csv(os.path.join("output_csv", f"Number_{short_name}.csv"), index=False)
    print(f"  -> output_csv/Volume_{short_name}.csv and Number_{short_name}.csv")